# CUDA Device Detection and CUDA Cores Counter

Detects NVIDIA GPUs and counts the actual number of CUDA cores.

## CUDA Core Counting Methodology

NVIDIA does **not** expose CUDA core counts through their APIs (PyTorch, CUDA Runtime,
nvidia-smi, etc.). These APIs only provide:

- Number of Streaming Multiprocessors (SMs)
- Compute Capability (e.g., 8.6, 10.0)

The CUDA cores per SM varies by architecture and must be looked up from NVIDIA's
public specifications. This is the standard approach used by all GPU monitoring tools.

| Architecture | Year | Cores/SM |
|---|---|---|
| Kepler | 2012 | 192 |
| Maxwell | 2014 | 128 |
| Pascal | 2016 | 64-128 |
| Volta/Turing | 2017-2018 | 64 |
| Ampere | 2020 | 64-128 |
| Ada Lovelace | 2022 | 128 |
| Hopper | 2022 | 128 |
| Blackwell | 2024 | 128 |

In [ ]:
import sys
import subprocess
import platform
import time

## Architecture Lookup Functions

These functions map compute capability to architecture name and CUDA cores per SM.

In [ ]:
def get_architecture_name(major, minor):
    """Get the architecture name based on compute capability."""
    arch_map = {
        3: "Kepler",
        5: "Maxwell",
        6: "Pascal",
        7: "Volta/Turing",
        8: "Ampere/Ada Lovelace",
        9: "Hopper",
        10: "Blackwell",
        11: "Blackwell",  # Mobile Blackwell variants
        12: "Blackwell",  # Mobile Blackwell variants
    }
    return arch_map.get(major, f"Unknown (SM {major}.{minor})")


def get_cuda_cores_per_sm(major, minor, device_name=""):
    """
    Get the number of CUDA cores per Streaming Multiprocessor (SM)
    based on the GPU's compute capability.

    Returns:
        tuple: (cores_per_sm, is_estimated)
    """
    cores_per_sm_dict = {
        # Kepler
        (3, 0): 192, (3, 5): 192, (3, 7): 192,
        # Maxwell
        (5, 0): 128, (5, 2): 128, (5, 3): 128,
        # Pascal
        (6, 0): 64, (6, 1): 128, (6, 2): 128,
        # Volta
        (7, 0): 64,
        # Turing
        (7, 5): 64,
        # Ampere
        (8, 0): 64, (8, 6): 128, (8, 7): 128,
        # Ada Lovelace
        (8, 9): 128,
        # Hopper
        (9, 0): 128,
        # Blackwell
        (10, 0): 128, (11, 0): 128, (12, 0): 128,
    }

    if (major, minor) in cores_per_sm_dict:
        return cores_per_sm_dict[(major, minor)], False

    estimated_cores = 128 if major >= 8 else 64
    return estimated_cores, True

## 1. System Information

In [ ]:
print("💻 System Information")
print("=" * 50)
print(f"Operating System: {platform.system()} {platform.release()}")
print(f"Python Version: {sys.version}")
print(f"Architecture: {platform.machine()}")

## 2. Check CUDA Installation (nvidia-smi)

In [ ]:
print("🔍 Checking CUDA Installation...")
print("=" * 50)

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print("✅ NVIDIA Driver detected")
        print("📊 GPU Information:")
        print(result.stdout)
        cuda_installed = True
    else:
        print("❌ NVIDIA Driver not found or not working")
        print("Error:", result.stderr)
        cuda_installed = False
except FileNotFoundError:
    print("❌ nvidia-smi command not found")
    print("   This usually means NVIDIA drivers are not installed")
    cuda_installed = False
except subprocess.TimeoutExpired:
    print("❌ nvidia-smi command timed out")
    cuda_installed = False
except Exception as e:
    print(f"❌ Error running nvidia-smi: {e}")
    cuda_installed = False

## 3. PyTorch CUDA Support and CUDA Core Count

In [ ]:
print("🔥 Checking PyTorch CUDA Support...")
print("=" * 50)

pytorch_cuda = False
try:
    import torch
    print(f"✅ PyTorch version: {torch.__version__}")

    if torch.cuda.is_available():
        print(f"✅ CUDA is available in PyTorch")
        print(f"📊 CUDA version: {torch.version.cuda}")
        print(f"🔢 Number of GPU devices detected: {torch.cuda.device_count()}")
        print()

        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            device_name = props.name
            device_capability = (props.major, props.minor)
            device_memory = props.total_memory / 1024**3
            sm_count = props.multi_processor_count
            arch_name = get_architecture_name(props.major, props.minor)

            cores_per_sm, is_estimated = get_cuda_cores_per_sm(props.major, props.minor, device_name)
            total_cuda_cores = sm_count * cores_per_sm

            print(f"   🎮 GPU {i}: {device_name}")
            print(f"      Architecture: {arch_name}")
            print(f"      Compute Capability: {device_capability[0]}.{device_capability[1]}")
            print(f"      Total Memory: {device_memory:.1f} GB")
            print(f"      Streaming Multiprocessors (SMs): {sm_count}")
            print(f"      CUDA Cores per SM: {cores_per_sm}")

            if is_estimated:
                print(f"      ⚠️  ESTIMATED Total CUDA Cores: ~{total_cuda_cores:,}")
                print(f"      ⚠️  Warning: Unknown architecture - this is an estimate!")
            else:
                print(f"      ⭐ Total CUDA Cores: {total_cuda_cores:,}")

            try:
                if hasattr(props, 'max_threads_per_block'):
                    print(f"      Max Threads per Block: {props.max_threads_per_block}")
                if hasattr(props, 'max_threads_per_multi_processor'):
                    print(f"      Max Threads per SM: {props.max_threads_per_multi_processor:,}")
            except Exception:
                pass

            print()

        pytorch_cuda = True
    else:
        print("❌ CUDA is not available in PyTorch")
        print("   PyTorch was likely installed without CUDA support")

except ImportError:
    print("❌ PyTorch is not installed")
    print("   Install with: pip install torch")
except Exception as e:
    print(f"❌ Error checking PyTorch CUDA: {e}")

## 4. TensorFlow CUDA Support

In [ ]:
print("🧠 Checking TensorFlow CUDA Support...")
print("=" * 50)

tensorflow_cuda = False
try:
    import tensorflow as tf
    print(f"✅ TensorFlow version: {tf.__version__}")

    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"✅ CUDA is available in TensorFlow")
        print(f"🔢 Number of GPU devices: {len(gpus)}")

        for i, gpu in enumerate(gpus):
            print(f"   GPU {i}: {gpu.name}")
            try:
                gpu_details = tf.config.experimental.get_device_details(gpu)
                if gpu_details:
                    compute_capability = gpu_details.get('compute_capability', 'Unknown')
                    print(f"     Compute Capability: {compute_capability}")
                    print(f"     Device Name: {gpu_details.get('device_name', 'Unknown')}")
            except Exception:
                pass

        tensorflow_cuda = True
    else:
        print("❌ No GPU devices found in TensorFlow")
        print("   TensorFlow was likely installed without CUDA support")

except ImportError:
    print("❌ TensorFlow is not installed")
    print("   Install with: pip install tensorflow")
except Exception as e:
    print(f"❌ Error checking TensorFlow CUDA: {e}")

## 5. CuPy CUDA Support

In [ ]:
print("☕ Checking CuPy CUDA Support...")
print("=" * 50)

cupy_cuda = False
try:
    import cupy as cp
    print(f"✅ CuPy version: {cp.__version__}")

    device_count = cp.cuda.runtime.getDeviceCount()
    print(f"✅ CUDA is available in CuPy")
    print(f"🔢 Number of CUDA devices: {device_count}")
    print()

    for i in range(device_count):
        with cp.cuda.Device(i):
            props = cp.cuda.runtime.getDeviceProperties(i)
            device_name = props['name'].decode()
            sm_count = props['multiProcessorCount']
            major = props['major']
            minor = props['minor']
            arch_name = get_architecture_name(major, minor)

            cores_per_sm, is_estimated = get_cuda_cores_per_sm(major, minor, device_name)
            total_cuda_cores = sm_count * cores_per_sm

            print(f"   🎮 GPU {i}: {device_name}")
            print(f"      Architecture: {arch_name}")
            print(f"      Compute Capability: {major}.{minor}")
            print(f"      Streaming Multiprocessors: {sm_count}")

            if is_estimated:
                print(f"      ⚠️  ESTIMATED Total CUDA Cores: ~{total_cuda_cores:,}")
            else:
                print(f"      ⭐ Total CUDA Cores: {total_cuda_cores:,}")
            print()

    cupy_cuda = True

except ImportError:
    print("❌ CuPy is not installed")
    print("   Install with: pip install cupy-cuda11x (or cupy-cuda12x)")
except Exception as e:
    print(f"❌ Error checking CuPy CUDA: {e}")

## 6. CUDA Computation Test

Runs a simple 1000x1000 matrix multiplication on GPU to verify CUDA works end-to-end.

In [ ]:
print("🧪 Running CUDA Computation Test...")
print("=" * 50)

if pytorch_cuda:
    try:
        device = torch.device('cuda')
        print(f"🔄 Testing computation on device: {device}")

        a = torch.randn(1000, 1000, device=device)
        b = torch.randn(1000, 1000, device=device)

        print("   Performing matrix multiplication (1000x1000)...")
        start_time = time.time()
        c = torch.matmul(a, b)
        torch.cuda.synchronize()
        end_time = time.time()

        print(f"✅ CUDA computation successful!")
        print(f"   Time taken: {end_time - start_time:.4f} seconds")
        print(f"   Result shape: {c.shape}")
        print(f"   Result device: {c.device}")
    except Exception as e:
        print(f"❌ Error during CUDA test: {e}")
else:
    print("⏭️  Skipping — PyTorch CUDA not available")

## 7. Summary

In [ ]:
print("📋 Summary")
print("=" * 50)
print(f"NVIDIA Driver: {'✅ Available' if cuda_installed else '❌ Not Available'}")
print(f"PyTorch CUDA: {'✅ Available' if pytorch_cuda else '❌ Not Available'}")
print(f"TensorFlow CUDA: {'✅ Available' if tensorflow_cuda else '❌ Not Available'}")
print(f"CuPy CUDA: {'✅ Available' if cupy_cuda else '❌ Not Available'}")

if pytorch_cuda:
    print(f"\n🎯 Total GPU Devices Found: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        cores_per_sm, is_estimated = get_cuda_cores_per_sm(props.major, props.minor, props.name)
        total_cores = props.multi_processor_count * cores_per_sm
        if is_estimated:
            print(f"   GPU {i} ({props.name}): ~{total_cores:,} CUDA Cores (ESTIMATED)")
        else:
            print(f"   GPU {i} ({props.name}): {total_cores:,} CUDA Cores")

print("\n🎯 Recommendations:")
if not cuda_installed:
    print("   • Install NVIDIA drivers from: https://www.nvidia.com/drivers/")
if not pytorch_cuda:
    print("   • Install PyTorch with CUDA: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")
if not tensorflow_cuda:
    print("   • Install TensorFlow with CUDA: pip install tensorflow[and-cuda]")
if not cupy_cuda:
    print("   • Install CuPy: pip install cupy-cuda11x (or cupy-cuda12x)")

print("\n✨ Done!")